# LELA60331 Week 10 Seminar: From two classes to many

So far every classifier we have built has chosen between **two** classes: positive or negative. But many classification problems have **more than two**. Sentiment, for example, could be **positive**, **negative** or **neutral**; and a dialogue system might need to recognise dozens of different requests. This is **multiclass classification**.

This week we see how to turn logistic regression into a multiclass classifier, called **multinomial logistic regression** or a **softmax classifier**, and how to measure how wrong it is. As in Weeks 5 and 7, we build it one small step at a time:

| Part | What you do |
|---|---|
| 1 | Look at a data set with three classes |
| 2 | Represent the labels as **one-hot vectors** |
| 3 | Calculate a **score for each class** |
| 4 | Turn the scores into probabilities with the **softmax** function |
| 5 | Choose the predicted class |
| 6 | Measure how wrong the predictions are: the **cross-entropy loss** |
| 7 | Put the classifier together from the parts in the LELA60341 Week 8 workbook |

Each activity has a cell to complete (replace every `???`) followed by a ✅ **check** cell. Run the check: ✅ means your answer is right, ❌ means not yet. Many activities also tell you what output you should get.

As before, we use **plain Python** (lists, loops, functions and list comprehensions) plus `math`, `random` and `matplotlib`.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used after each activity.
# You don't need to understand how it works: it prints ✅ when your answer is right and ❌ when it isn't yet.
def check(label, condition):
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

This cell sets up `dot` (Week 4) and `mat_vec` (Week 7). Run it.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def dot(a, b):
    # the dot product of two lists of the same length (Week 4)
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # matrix-vector multiplication: the dot product of each row of the matrix with the vector (Week 7)
    result = []
    for row in matrix:
        result.append(dot(row, vector))
    return result

---
# Part 1 · Three classes

Here is a simulated data set for a three-class sentiment problem. As in Weeks 5 and 7, each text has two features: the log count of positive words (`x1`) and the log count of negative words (`x2`). Each text's features are a row of the matrix `X`, as in Week 7, Part 4.

* **positive** texts (class 0) have many positive words and few negative ones
* **negative** texts (class 1) have the reverse
* **neutral** texts (class 2) have either few of both or many of both

The classes are numbered 0, 1 and 2, and `class_names[c]` is the name of class `c`. `labels[k]` is the class of text `k`. Run the cells to make the data and plot it.

In [ ]:
# the simulated three-class data
random.seed(10)
X = []          # one row [x1, x2] per text: the log counts of positive and of negative words
labels = []     # the class of each text: 0 = positive, 1 = negative, 2 = neutral
for i in range(20):                    # positive texts: many positive words, few negative ones
    X.append([random.gauss(3, 0.5), random.gauss(1, 0.5)])
    labels.append(0)
for i in range(20):                    # negative texts: the reverse
    X.append([random.gauss(1, 0.5), random.gauss(3, 0.5)])
    labels.append(1)
for i in range(10):                    # neutral texts: few of both ...
    X.append([random.gauss(1, 0.5), random.gauss(1, 0.5)])
    labels.append(2)
for i in range(10):                    # ... or many of both
    X.append([random.gauss(3, 0.5), random.gauss(3, 0.5)])
    labels.append(2)
class_names = ["positive", "negative", "neutral"]
num_samples = len(labels)
num_classes = 3

print(len(X), "texts")
print(X[0], labels[0], class_names[labels[0]])

In [ ]:
def plot_data():
    markers = ["o", "*", "x"]
    for c in range(num_classes):
        xs = [X[k][0] for k in range(num_samples) if labels[k] == c]
        ys = [X[k][1] for k in range(num_samples) if labels[k] == c]
        plt.scatter(xs, ys, marker=markers[c], s=60, label=class_names[c])
    plt.xlabel("x1: log count of positive words")
    plt.ylabel("x2: log count of negative words")
    plt.legend()

plot_data()
plt.show()

Can a single straight line separate the three classes? No: we need a way of scoring **each** class.

### Activity 1 — Getting to know the data

1. `class_counts`: a list with the number of texts in each class, in the order 0, 1, 2. *Hint:* the list method `count` (Week 2 Python workbook) counts how often a value occurs in a list: `labels.count(0)` is the number of positive texts.
2. `first_negative`: the position `k` of the first negative text (class 1). *Hint:* the list method `index` gives the position of the first occurrence of a value.
3. `first_negative_features`: that text's row of `X`

In [ ]:
class_counts = [???, ???, ???]
first_negative = ???
first_negative_features = ???
print(class_counts, first_negative, first_negative_features)     # you should get [20, 20, 20] 20 and then text 20's features

In [ ]:
# ✅ Check
check("class_counts", lambda: class_counts == [20, 20, 20])
check("first_negative", lambda: first_negative == 20)
check("first_negative_features", lambda: first_negative_features == X[20])

Does text 20's row fit what you'd expect for a negative text?

---
# Part 2 · Labels as one-hot vectors

With two classes, each label was a single number, 0 or 1, and the model gave a single probability: the probability of class 1. With three classes, the model will give **three** probabilities, one per class. To compare them with the correct answer, we write the correct answer as three numbers too: a **one-hot vector**, which is all 0s except for a single 1 in the position of the correct class.

| class | one-hot vector |
|---|---|
| 0 positive | `[1, 0, 0]` |
| 1 negative | `[0, 1, 0]` |
| 2 neutral | `[0, 0, 1]` |

You can read a one-hot vector as a set of probabilities: the correct class has probability 1, and every other class has probability 0.

### Activity 2 — One-hot vectors

1. Complete `one_hot(c, n_classes)`, which returns the one-hot vector for class `c` out of `n_classes` classes. It starts with a list of `n_classes` 0s; then set the element at position `c` to 1.
2. `Y`: the one-hot vector of every text, in order. *Hint:* a list comprehension over `labels`.

In [ ]:
def one_hot(c, n_classes):
    vector = [0] * n_classes          # a list of n_classes 0s, e.g. [0, 0, 0]
    ???                   # put a 1 at position c
    return vector

print(one_hot(0, 3), one_hot(2, 3), one_hot(1, 5))     # you should get [1, 0, 0] [0, 0, 1] [0, 1, 0, 0, 0]

Y = ???
print(len(Y), Y[0], Y[20], Y[59])                      # you should get 60 [1, 0, 0] [0, 1, 0] [0, 0, 1]

In [ ]:
# ✅ Check
check("one_hot", lambda: one_hot(0, 3) == [1, 0, 0] and one_hot(2, 3) == [0, 0, 1] and one_hot(1, 5) == [0, 1, 0, 0, 0])
check("a new list each time", lambda: one_hot(0, 3) is not one_hot(0, 3))
check("Y", lambda: len(Y) == 60 and Y[0] == [1, 0, 0] and Y[20] == [0, 1, 0] and Y[59] == [0, 0, 1])

---
# Part 3 · A score for each class

In logistic regression, each text got **one** score, `z = bias + x1 * weight1 + x2 * weight2`, which said how strongly the model thought the text was positive.

In a softmax classifier, each **class** has its own weights and its own bias, so each text gets **one score per class**. The score for class `c` says how strongly the model thinks the text belongs to class `c`:

`z[c] = b[c] + x1 * W[c][0] + x2 * W[c][1]`

We store the weights as a **matrix** `W` with **one row per class** and one column per feature, and the biases as a list `b` with one bias per class. Here is a model that has already been trained:

In [ ]:
#       weight for x1, weight for x2
W = [[ 1.80, -0.75],        # class 0: positive
     [-0.70,  1.81],        # class 1: negative
     [ 0.30,  0.30]]        # class 2: neutral
b = [-0.45, -0.82, 0.50]    # one bias per class

Look at the weights. For the positive class, positive words (`x1`) push the score **up** and negative words push it **down**. What about the other two classes?

### Activity 3 — The scores of one text, class by class

Row `c` of `W` holds class `c`'s weights, so the score for class `c` is the **dot product** of row `c` with the text's features, plus `b[c]`. For text 0 (`X[0]`), calculate:

1. `z0`: the score for class 0, using `dot` with `W[0]` and `X[0]`, plus `b[0]`
2. `z1` and `z2`: the same for classes 1 and 2

In [ ]:
z0 = ???
z1 = ???
z2 = ???
print(round(z0, 3), round(z1, 3), round(z2, 3))      # you should get 3.514 -1.192 1.488

In [ ]:
# ✅ Check
check("z0", lambda: abs(z0 - (b[0] + X[0][0] * W[0][0] + X[0][1] * W[0][1])) < 1e-9)
check("z1", lambda: abs(z1 - (b[1] + X[0][0] * W[1][0] + X[0][1] * W[1][1])) < 1e-9)
check("z2", lambda: abs(z2 - (b[2] + X[0][0] * W[2][0] + X[0][1] * W[2][1])) < 1e-9)

Text 0 is positive. Which class got the highest score?

### All the classes at once

Taking the dot product of **each row** of `W` with the features is exactly **matrix-vector multiplication** (Week 7, Part 4): `mat_vec(W, x)` gives the scores of all the classes at once (without the biases), one number per row of `W`, that is, one per class.

### Activity 4 — A `scores` function

Complete `scores(W, b, x)`, which returns the list of scores of all the classes for the features `x`:

1. `z`: `mat_vec` with `W` and `x`
2. complete the loop so that it adds each class's bias to its score

In [ ]:
def scores(W, b, x):
    z = ???                 # 1. one number per class
    for c in range(len(z)):
        z[c] += ???                  # 2. add class c's bias
    return z

z = scores(W, b, X[0])
print([round(value, 3) for value in z])          # the same three scores as in Activity 3

In [ ]:
# ✅ Check
check("the scores of text 0", lambda: all(abs(u - v) < 1e-9 for u, v in zip(scores(W, b, X[0]), [z0, z1, z2])))
check("a small example", lambda: scores([[1, 0], [0, 1], [1, 1]], [0, 10, 100], [2, 3]) == [2, 13, 105])

---
# Part 4 · From scores to probabilities: softmax

In logistic regression we turned the score into a probability with the **sigmoid** function. Now we have a score for each class, and we want a probability for each class. These probabilities must all be between 0 and 1, and they must **add up to 1**, because the text belongs to exactly one of the classes.

The **softmax** function does this in two steps:

1. **exponentiate** each score: $e^{z_c}$, with `math.exp`. This makes every number positive, and a bigger score gives a bigger number.
2. **divide** each of these by their total, so that they add up to 1

$$q_c = \frac{e^{z_c}}{e^{z_0} + e^{z_1} + e^{z_2}}$$

In words: each class gets a **share** of the total, and a class with a bigger score gets a bigger share. Here are the two steps for the scores `[2.0, 1.0, 0.1]`:

| | class 0 | class 1 | class 2 | total |
|---|---|---|---|---|
| score `z` | 2.0 | 1.0 | 0.1 | |
| `math.exp(z)` | 7.389 | 2.718 | 1.105 | 11.212 |
| divided by the total | 0.659 | 0.242 | 0.099 | 1.0 |

### Activity 5 — The softmax function

Complete `softmax(z)`:

1. `exps`: complete the loop so that it adds `math.exp` of each score to the list
2. `total`: the sum of `exps`
3. `probs`: complete the loop so that it adds each element of `exps` divided by `total`

In [ ]:
def softmax(z):
    exps = []
    for value in z:
        exps.append(???)      # 1.
    total = ???                     # 2.
    probs = []
    for e in exps:
        probs.append(???)           # 3.
    return probs

p = softmax([2.0, 1.0, 0.1])
print([round(value, 3) for value in p], sum(p))      # you should get [0.659, 0.242, 0.099] and 1.0 (or 0.9999999999999999)

In [ ]:
# ✅ Check
check("the example", lambda: [round(v, 3) for v in softmax([2.0, 1.0, 0.1])] == [0.659, 0.242, 0.099])
check("adds up to 1", lambda: abs(sum(softmax([5, -3, 0.5, 2])) - 1) < 1e-12)
check("equal scores give equal shares", lambda: softmax([1.5, 1.5]) == [0.5, 0.5])

🔮 **Predict**, then try it: what does `softmax([12.0, 11.0, 10.1])` give? (Each score is 10 more than in the example.) What about `softmax([4.0, 2.0, 0.2])`, where each score is doubled?

In [ ]:
print([round(value, 3) for value in softmax([12.0, 11.0, 10.1])])
print([round(value, 3) for value in softmax([4.0, 2.0, 0.2])])

Adding the same number to every score changes nothing: only the **differences** between the scores matter. Doubling the scores makes the probabilities more extreme.

### Activity 6 — Class probabilities for every text

Use `scores` and `softmax` with the model `W` and `b` from Part 3:

1. `q0`: the probabilities of the three classes for text 0
2. `Q`: a list with the probabilities for **every** text (one list of three probabilities per text). *Hint:* a list comprehension over the rows of `X`.

In [ ]:
q0 = ???
Q = ???
print([round(value, 3) for value in q0])         # you should get [0.876, 0.008, 0.116]
print(len(Q), [round(value, 3) for value in Q[20]])

In [ ]:
# ✅ Check
check("q0", lambda: [round(v, 3) for v in q0] == [0.876, 0.008, 0.116])
check("Q", lambda: len(Q) == 60 and all(abs(sum(q) - 1) < 1e-9 for q in Q))
check("Q[20]", lambda: Q[20] == softmax(scores(W, b, X[20])))

---
# Part 5 · The predicted class

The model predicts the class with the **highest** probability. We need its **position** in the list: for `[0.1, 0.7, 0.2]` that is 1. (Because softmax keeps the order of the scores, the class with the highest probability is also the class with the highest score.)

### Activity 7 — The position of the largest value

Complete `argmax(values)`, which returns the position of the largest number in `values`. It goes through the positions one by one, keeping in `best` the position of the largest number seen so far. Fill in the condition: is the number at position `c` bigger than the number at position `best`?

In [ ]:
def argmax(values):
    best = 0
    for c in range(len(values)):
        if ???:
            best = c
    return best

print(argmax([0.1, 0.7, 0.2]), argmax([5, 3, 9, 1]), argmax([0.6, 0.3, 0.1]))     # you should get 1 2 0

In [ ]:
# ✅ Check
check("argmax", lambda: argmax([0.1, 0.7, 0.2]) == 1 and argmax([5, 3, 9, 1]) == 2 and argmax([0.6, 0.3, 0.1]) == 0)
check("the last element", lambda: argmax([1, 2, 3]) == 2)

### Activity 8 — Predictions and accuracy

1. `predictions`: the predicted class of every text, using `argmax` on each list of probabilities in `Q`
2. `accuracy`: the proportion of texts whose prediction equals their label (as in Week 5)

In [ ]:
predictions = ???
accuracy = ???
print(predictions)
print("accuracy:", accuracy)       # you should get 0.75

In [ ]:
# ✅ Check
check("predictions", lambda: len(predictions) == 60 and predictions[:5] == [argmax(q) for q in Q[:5]])
check("accuracy", lambda: abs(accuracy - 0.75) < 1e-9)

### Activity 9 — Classifying new texts

Complete `predict_sentiment(n_positive, n_negative)`, which returns the predicted class **name** for a text with the given **counts** of positive and negative words. Remember that the features are the **logs** of the counts.

What does the model predict for these texts? Do the predictions make sense?

a) 10 positive words and 3 negative words
b) 3 positive words and 3 negative words
c) 1 positive word and 6 negative words

In [ ]:
def predict_sentiment(n_positive, n_negative):
    x = ???
    q = ???
    return ???

print(predict_sentiment(10, 3), predict_sentiment(3, 3), predict_sentiment(1, 6))     # you should get positive neutral negative

In [ ]:
# ✅ Check
check("a) 10 positive, 3 negative", lambda: predict_sentiment(10, 3) == "positive")
check("b) 3 positive, 3 negative", lambda: predict_sentiment(3, 3) == "neutral")
check("c) 1 positive, 6 negative", lambda: predict_sentiment(1, 6) == "negative")

---
# Part 6 · Measuring how wrong the model is: the cross-entropy loss

In Week 5, the loss for one text was **minus the log of the probability the model gave to the correct answer**. That works for any number of classes:

`loss = -log2(q[c])`, where `c` is the correct class

If the model gives the correct class a probability of 1, the loss is 0; the lower the probability it gives the correct class, the bigger the loss. Here are two examples for a text whose correct class is 0:

| probabilities `q` | probability of the correct class | loss |
|---|---|---|
| `[0.7, 0.2, 0.1]` | 0.7 | 0.515 |
| `[0.1, 0.2, 0.7]` | 0.1 | 3.322 |

### Activity 10 — The loss for one text

1. Complete `loss_one(q, c)`, which returns the loss for one text whose probabilities are `q` and whose correct class is `c`. Use `math.log2`.
2. The same loss can be written with the one-hot vector `y`: $-\sum_c y_c \log_2 q_c$. Every term except the correct class's is 0 (because `y[c]` is 0), so it gives the same answer. Complete the loop in `loss_one_hot(q, y)`, which adds up `-y[c] * math.log2(q[c])` over the classes.

In [ ]:
def loss_one(q, c):
    return ???                          # 1.

def loss_one_hot(q, y):
    total = 0
    for c in range(len(q)):
        total += ???               # 2.
    return total

print(round(loss_one([0.7, 0.2, 0.1], 0), 3), round(loss_one([0.1, 0.2, 0.7], 0), 3))      # you should get 0.515 3.322
print(round(loss_one_hot([0.7, 0.2, 0.1], [1, 0, 0]), 3))                                  # the same as the first: 0.515

In [ ]:
# ✅ Check
check("loss_one", lambda: round(loss_one([0.7, 0.2, 0.1], 0), 3) == 0.515 and round(loss_one([0.1, 0.2, 0.7], 0), 3) == 3.322)
check("loss_one for another class", lambda: round(loss_one([0.1, 0.2, 0.7], 2), 3) == 0.515)
check("loss_one_hot gives the same", lambda: abs(loss_one_hot([0.1, 0.2, 0.7], [0, 0, 1]) - loss_one([0.1, 0.2, 0.7], 2)) < 1e-12)

### Activity 11 — The loss for all the texts

As in Week 5, the loss for the whole data set is the **average** of the losses for all the texts.

1. Complete `average_loss(W, b, X, labels)`, which returns the average loss of the model with weights `W` and biases `b` over all the texts. *Hint:* for each text, calculate its probabilities with `scores` and `softmax`, then its loss with `loss_one`.
2. `trained_loss`: the average loss of the trained model from Part 3
3. `untrained_loss`: the average loss of an **untrained** model with random weights and biases (made for you)

Which loss is lower? Why?

In [ ]:
def average_loss(W, b, X, labels):
    losses = []
    for k in range(len(labels)):
        q = ???
        losses.append(???)
    return ???

trained_loss = ???

random.seed(1)
W_random = [[random.random(), random.random()] for c in range(num_classes)]
b_random = [0, 0, 0]
untrained_loss = ???
print(round(trained_loss, 3), round(untrained_loss, 3))      # you should get 0.676 2.325

In [ ]:
# ✅ Check
check("trained_loss", lambda: abs(trained_loss - 0.676) < 0.001)
check("untrained_loss", lambda: abs(untrained_loss - 2.325) < 0.001)

The untrained model's loss is higher: its probabilities for the correct classes are lower. Training a softmax classifier means finding the weights and biases that make this average loss as small as possible, by gradient descent, just as for logistic regression.

---
# Part 7 · The classifier as a model

In the LELA60341 Week 8 workbook you built a logistic regression model out of parts: `Parameter`, `Module`, a `Linear` layer, a loss and an optimiser. Look again at what the `Linear` layer does: it has a weight matrix with **one row per output** and one bias per output, and for each example it gives `mat_vec(weight, x)` plus the biases. That is exactly our `scores` function, with one output per **class**.

So a softmax classifier for 2 features and 3 classes is simply `Linear(2, 3)`, and the only new part we need is a loss for several classes. Run the cell to set up the parts from LELA60341 Week 8.

In [ ]:
def transpose(matrix):
    # the rows of the matrix turned into columns
    return [[row[j] for row in matrix] for j in range(len(matrix[0]))]

def zeros_like(value):
    # a list or matrix of 0s with the same shape as value
    if isinstance(value[0], list):
        return [[0] * len(row) for row in value]
    else:
        return [0] * len(value)

class Parameter:
    """Numbers that training changes: their values, and their gradients."""
    def __init__(self, value):
        self.value = value
        self.grad = zeros_like(value)

class Module:
    """The parent class of every layer and every model."""
    def __init__(self):
        pass

    def __call__(self, X):
        return self.forward(X)

    def parameters(self):
        params = []
        for value in vars(self).values():
            if isinstance(value, Parameter):
                params.append(value)
            elif isinstance(value, Module):
                params.extend(value.parameters())
        return params

class Linear(Module):
    """A layer that gives n_out scores for each example: one row of weights and one bias per output."""
    def __init__(self, n_in, n_out):
        super().__init__()
        self.weight = Parameter([[random.random() for j in range(n_in)] for c in range(n_out)])
        self.bias = Parameter([random.random() for c in range(n_out)])

    def forward(self, X):
        self._X = X
        out = []
        for x in X:
            z = mat_vec(self.weight.value, x)
            for c in range(len(z)):
                z[c] += self.bias.value[c]
            out.append(z)
        return out

    def backward(self, grad_out):
        W = self.weight.value
        for k in range(len(self._X)):
            for c in range(len(W)):
                for j in range(len(W[c])):
                    self.weight.grad[c][j] += grad_out[k][c] * self._X[k][j]
                self.bias.grad[c] += grad_out[k][c]
        WT = transpose(W)
        return [mat_vec(WT, grad_out[k]) for k in range(len(self._X))]

class SGD:
    """Gradient descent: new value = old value - learning rate * gradient."""
    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = zeros_like(p.value)

    def step(self):
        for p in self.parameters:
            if isinstance(p.value[0], list):
                for i in range(len(p.value)):
                    for j in range(len(p.value[i])):
                        p.value[i][j] -= self.lr * p.grad[i][j]
            else:
                for i in range(len(p.value)):
                    p.value[i] -= self.lr * p.grad[i]

### Activity 12 — A softmax classifier is a `Linear` layer

1. `model`: a `Linear` layer with 2 inputs and 3 outputs. Then set its weights and biases to the trained `W` and `b` from Part 3 (done for you).
2. `all_scores`: the scores of every text, using `model` on `X`
3. `model_predictions`: the predicted class of every text, with `argmax` on each row of `all_scores`. (Softmax keeps the order of the scores, so the class with the highest score is the class with the highest probability.)

In [ ]:
model = ???                    # 1.
model.weight.value = W
model.bias.value = b

all_scores = ???                   # 2.
model_predictions = ???    # 3.
print([round(v, 3) for v in all_scores[0]])      # the same scores as in Activity 3
print(model_predictions == predictions)          # you should get True

In [ ]:
# ✅ Check
check("1. a Linear layer with 3 outputs", lambda: isinstance(model, Linear) and len(model.weight.value) == 3)
check("2. the same scores as scores()", lambda: all(abs(all_scores[k][c] - scores(W, b, X[k])[c]) < 1e-12 for k in range(60) for c in range(3)))
check("3. the same predictions", lambda: model_predictions == predictions)

### Activity 13 — A `CrossEntropyLoss` class

In LELA60341 Week 8, the loss was an object: calling it with the scores and the labels returns the average loss. Complete `CrossEntropyLoss`, which does the same for several classes. Called as `loss_fn(scores, labels)`, it:

1. turns each row of scores into probabilities with `softmax`, and keeps them in `self._Q` (they are needed for the gradients)
2. calculates the loss for each text with `loss_one`
3. returns the average

Its loss for our model should be the same as `trained_loss` from Activity 11.

In [ ]:
class CrossEntropyLoss:
    """The cross-entropy loss for several classes, from the scores."""
    def __call__(self, scores, labels):
        self._Q = ???                                # 1.
        self._labels = labels
        losses = ???    # 2.
        return ???                                           # 3.

loss_fn = CrossEntropyLoss()
print(round(loss_fn(model(X), labels), 3), round(trained_loss, 3))      # the two numbers should be the same: 0.676

In [ ]:
# ✅ Check
check("the same loss as average_loss", lambda: abs(CrossEntropyLoss()(model(X), labels) - trained_loss) < 1e-12)
def _kept():
    f = CrossEntropyLoss()
    f(model(X[:2]), labels[:2])
    return len(f._Q) == 2 and abs(sum(f._Q[0]) - 1) < 1e-12
check("the probabilities are kept", _kept)

---
## ✏️ Summary

You have built every part of a softmax classifier except its training:

1. **one-hot vectors** represent the correct class as a list of 0s with a single 1
2. each class has its own row of weights in `W` and its own bias in `b`, so each text gets **one score per class**: `mat_vec(W, x)` plus the biases
3. **softmax** turns the scores into probabilities that add up to 1: exponentiate, then divide by the total
4. the **predicted class** is the one with the highest probability: `argmax`
5. the **cross-entropy loss** is minus the log of the probability given to the correct class, averaged over the texts
6. a softmax classifier is a `Linear` layer with **one output per class**, and its loss can be an object, `CrossEntropyLoss`, like the losses in the LELA60341 Week 8 workbook